# 🚪 AI Agents + MCP-Gateway — ein Tor für viele Server (mit LiteLLM & Docker Compose)

Dieses Notebook baut auf **`AI_Agents_MCP.ipynb`** auf. Dort sprach unser Client **direkt** mit einem MCP-Server (stdio). In der Praxis hast du aber schnell **viele** Server: einen für die Datenbank, einen für GitHub, einen fürs Filesystem … Jeden einzeln verbinden und absichern? Mühsam.

Ein **MCP-Gateway** löst das: **ein** Endpoint, der **mehrere** Upstream-MCP-Server bündelt — mit **zentraler Authentifizierung** (per API-Key/Team) und **Namespacing** der Tools. Der **[LiteLLM-Proxy](https://docs.litellm.ai/docs/mcp)** kann genau das.

**Der rote Faden bleibt:** *derselbe* Agent-Loop wie im MCP-Notebook. Es ändert sich nur **wohin** und **womit** wir uns verbinden:

| MCP-Notebook (direkt) | Hier (über LiteLLM) |
|---|---|
| Client → **direkt** zum Server | Client → **LiteLLM** → Server |
| Transport: **stdio** (Subprozess) | Transport: **streamable-http** (Netzwerk) |
| Keine Auth | **API-Key** (`x-litellm-api-key`) |
| Tool heißt `add` | Tool heißt `demo-add` (Alias-**Präfix**) |
| 1 Server pro Verbindung | **n Server** hinter *einem* Endpoint |

![image](images/mcp_gateway.png)

```
  Notebook (MCP-Client)
       │  streamable-http  +  x-litellm-api-key
       ▼
  ┌─────────────────── docker compose ───────────────────┐
  │  litellm  (:4000)  ── MCP-Gateway ──►  demo-mcp (:8000)│
  │     /mcp                                 unsere Tools  │
  └───────────────────────────────────────────────────────┘
```

**Neu gegenüber der ersten Fassung:** Die ganze Infrastruktur (unser HTTP-Server **und** LiteLLM) steckt jetzt in einer **`docker-compose.yml`** statt in Notebook-Code. Das Notebook macht nur noch `compose up` / `compose down` — alles Prozess-Gefrickel (Subprozess starten, Ports abwarten, alte Prozesse killen) fällt weg.

## 0 · Setup

Wie im MCP-Notebook: `llm()` als einziger Draht zum Modell (Azure OpenAI aus der `.env`) plus ein paar Helfer. Zusätzlich brauchst du **Docker Desktop**. Die Tool-Definitionen liegen in `mcp_demo_server.py` (dieselben wie im MCP-Notebook) und werden vom Compose-Service `demo-mcp` über HTTP exponiert.

In [1]:
import os, sys, json, asyncio, threading, subprocess, time, urllib.request
from pathlib import Path
from openai import AzureOpenAI
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

load_dotenv()

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)
DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]

def llm(messages, tools=None, tool_choice="auto"):
    """Unser einziger Draht zum Modell - identisch zum MCP-Notebook."""
    kwargs = dict(model=DEPLOYMENT, messages=messages)
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs)

def run_async(coro_factory):
    """Async-Funktion in eigenem Event-Loop in eigenem Thread ausfuehren.
    Auf Windows ein ProactorEventLoop - unabhaengig von Jupyters Loop im Hauptthread."""
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["v"] = loop.run_until_complete(coro_factory())
        except BaseException as e:
            box["e"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker, daemon=True)
    t.start(); t.join()
    if "e" in box:
        raise box["e"]
    return box["v"]

def to_openai_tools(mcp_tools):
    """MCP-Tool-Schema -> OpenAI-Tool-Format (das inputSchema IST bereits ein JSON-Schema)."""
    return [{"type": "function", "function": {
        "name": t.name,
        "description": t.description or "",
        "parameters": t.inputSchema or {"type": "object", "properties": {}},
    }} for t in mcp_tools]

def _mcp_text(result):
    """MCP-Tool-Ergebnis -> Text fuers Modell."""
    parts = [c.text for c in result.content if getattr(c, "type", None) == "text"]
    return "\n".join(parts) if parts else str(result.content)

print("Setup bereit. Deployment:", DEPLOYMENT)

Setup bereit. Deployment: gpt-5.4-mini


## 1 · Die Infrastruktur als Code

Statt im Notebook Prozesse zu starten, beschreiben wir die Infrastruktur **deklarativ** in ein paar kleinen Dateien:

| Datei | Rolle |
|---|---|
| `mcp_http_server.py` | startet **dieselben** Tools wie im MCP-Notebook über **streamable-http** (Transport-Wechsel, sonst unverändert) |
| `Dockerfile.mcp` | packt Python + `mcp`-SDK + unsere zwei Server-Dateien in ein winziges Image |
| `litellm_config.yaml` | registriert den Demo-Server als Upstream (`url: http://demo-mcp:8000/mcp`) und setzt den `master_key` |
| `docker-compose.yml` | startet die Services `demo-mcp` + `litellm` + `db` in **einem** Netz |

Der Clou: Im Compose-Netz erreicht LiteLLM den Demo-Server einfach über den **Service-Namen** `demo-mcp` — kein `host.docker.internal`, kein Port-Mapping für den Demo-Server nötig. Nur das Gateway (`:4000`) ist vom Host (= Notebook) aus sichtbar.

> 🖥️ **Admin-UI (optional):** Unter **http://localhost:4000/ui** (Login `admin` / `sk-1234`) zeigt LiteLLM eine UI für Keys, Teams, Logs und die registrierten MCP-Server. Dafür ist der `db`-Service (Postgres) da — die UI speichert in einer DB (sonst `Not connected to DB!` beim Login). Für den reinen Tool-Zugriff in diesem Notebook brauchst du die UI **nicht**.

Werfen wir kurz einen Blick auf die zentrale Datei:

In [2]:
print("=== docker-compose.yml ===\n")
print(Path("docker-compose.yml").read_text(encoding="utf-8"))
print("=== litellm_config.yaml (Auszug: der Upstream) ===\n")
print(Path("litellm_config.yaml").read_text(encoding="utf-8"))

=== docker-compose.yml ===

# MCP-Gateway-Infrastruktur fuer das Notebook AI_Agents_MCP_Gateway.ipynb.
#
# Drei Services in einem Netz:
#   db       : Postgres - NUR fuer die LiteLLM-Admin-UI noetig (User/Keys/Logs)
#   demo-mcp : unser Demo-Server (current_time, add, search_kb) ueber streamable-http
#   litellm  : der LiteLLM-Proxy als MCP-Gateway, erreichbar auf http://localhost:4000
#
# LiteLLM spricht den Demo-Server ueber den Service-Namen 'demo-mcp' an (siehe
# litellm_config.yaml) - deshalb braucht der Demo-Server kein Port-Mapping zum Host.
#
# Der Microsoft-365-Server (ms365) laeuft NICHT als eigener Service, sondern wird
# von LiteLLM als stdio-Subprozess gestartet (siehe litellm_config.yaml). Dafuer
# enthaelt das LiteLLM-Image Node (Dockerfile.litellm). Der Login-Token wird im
# Named Volume 'ms365_token' gehalten, damit der Device-Code-Login Neustarts ueberlebt.
#
# Admin-UI:  http://localhost:4000/ui   (Login: admin / sk-1234)
# Starten:   docker compose up -d --build    

## 2 · Gateway starten — `docker compose up`

Eine Zeile fährt **alle** Container hoch (`db`, `demo-mcp`, `litellm`). Die Zelle wartet anschließend, bis das Gateway über `/health/readiness` antwortet.

> ⏳ Der **erste** Start baut das Mini-Image und zieht die LiteLLM-/Postgres-Images (~mehrere hundert MB) — das kann ein paar Minuten dauern. Danach geht es schnell.

In [3]:
GATEWAY_URL     = "http://localhost:4000/mcp"
GATEWAY_HEADERS = {"x-litellm-api-key": "Bearer sk-1234"}   # = master_key aus litellm_config.yaml

def _ready():
    try:
        with urllib.request.urlopen("http://localhost:4000/health/readiness", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def _wait(cond, what, timeout=300, every=2):
    start = time.time()
    while time.time() - start < timeout:
        if cond():
            print(f"✓ {what} bereit"); return True
        time.sleep(every)
    print(f"✗ {what} nicht bereit nach {timeout}s"); return False

# Defensive: evtl. alten Standalone-Container aus der ersten Fassung entfernen (haelt sonst :4000).
subprocess.run(["docker", "rm", "-f", "litellm-mcp-gateway"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starte das Gateway via docker compose ... (erster Start baut/zieht die Images)")
up = subprocess.run(["docker", "compose", "up", "-d", "--build"], capture_output=True, text=True)
print(up.stdout.strip() or up.stderr.strip())

_wait(_ready, "LiteLLM-Gateway (:4000)", timeout=300)
print("\n--- Status ---")
print(subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True).stdout)

Starte das Gateway via docker compose ... (erster Start baut/zieht die Images)
#1 [internal] load local bake definitions
#1 reading from stdin 1.13kB done
#1 DONE 0.0s

#2 [demo-mcp internal] load build definition from Dockerfile.mcp
#2 transferring dockerfile: 455B done
#2 DONE 0.0s

#3 [litellm internal] load build definition from Dockerfile.litellm
#3 transferring dockerfile: 481B done
#3 DONE 0.0s

#4 [litellm internal] load metadata for ghcr.io/berriai/litellm-database:main-latest
#4 DONE 0.1s

#5 [demo-mcp internal] load metadata for docker.io/library/python:3.12-slim
#5 ...

#6 [litellm internal] load .dockerignore
#6 transferring context: 2B done
#6 DONE 0.0s

#7 [litellm 1/2] FROM ghcr.io/berriai/litellm-database:main-latest@sha256:0e8dfd691ceeb660dc43165bb6205ea92921b2cf80f08f4bfd2efd237c23b938
#7 resolve ghcr.io/berriai/litellm-database:main-latest@sha256:0e8dfd691ceeb660dc43165bb6205ea92921b2cf80f08f4bfd2efd237c23b938 0.0s done
#7 DONE 0.0s

#8 [litellm 2/2] RUN npm install

## 3 · Verbinden — derselbe Client, neuer Transport

Statt `stdio_client(...)` benutzen wir `streamablehttp_client(url, headers=...)` aus dem **gleichen** `mcp`-SDK. Alles danach — `ClientSession`, `initialize()`, `list_tools()`, `call_tool()` — ist **identisch**. Den `master_key` schicken wir als `x-litellm-api-key` mit.

Achte auf die **Tool-Namen**: Das Gateway präfixt sie mit dem Server-Alias (`demo-…`). Unser Agent-Loop ist davon **nicht** betroffen, weil er die Namen aus `list_tools()` **entdeckt** statt sie hart zu kodieren.

In [14]:
async def _gw_list():
    async with streamablehttp_client(GATEWAY_URL, headers=GATEWAY_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()                 # gleicher Handshake
            return (await session.list_tools()).tools  # gleiche Discovery - jetzt vom Gateway

gateway_tools = run_async(_gw_list)
print("Tools über das LiteLLM-Gateway (Alias-Präfix 'demo-'):")
for t in gateway_tools:
    print(f"  \U0001f50c {t.name}: {t.description}")

GATEWAY_OPENAI_TOOLS = to_openai_tools(gateway_tools)
print("\nBeispiel-Schema:", json.dumps(GATEWAY_OPENAI_TOOLS[0]["function"], ensure_ascii=False)[:200], "...")

Tools über das LiteLLM-Gateway (Alias-Präfix 'demo-'):
  🔌 demo-current_time: Gibt das aktuelle Datum und die Uhrzeit des Servers im ISO-Format zurück.
  🔌 demo-add: Addiert zwei Zahlen und gibt die Summe zurück.
  🔌 demo-search_kb: Durchsucht die Wissensdatenbank des Servers nach einem Stichwort und gibt passende Einträge zurück.
  🔌 ms365-login: Authenticate with Microsoft account
  🔌 ms365-logout: Log out from Microsoft account
  🔌 ms365-verify-login: Check current Microsoft authentication status
  🔌 ms365-list-accounts: List all Microsoft accounts configured in this server. Use this to discover available account emails before making tool calls. Reflects accounts added mid-session via --login.
  🔌 ms365-select-account: Select a Microsoft account as the default. Accepts email address (e.g. user@outlook.com) or account ID. Use list-accounts to discover available accounts.
  🔌 ms365-remove-account: Remove a Microsoft account from the cache. Accepts email address (e.g. user@outlook.com)

## 4 · Der Agent — Zeile für Zeile derselbe Loop, jetzt über das Gateway

**Dieselbe** Schleife wie im MCP-Notebook. Die zwei `# <- Gateway`-Stellen sind identisch zu vorher — nur die Verbindung läuft über `streamablehttp_client` zu LiteLLM. `session.call_tool(name, args)` ruft das Tool über das Gateway auf, LiteLLM reicht es an den `demo`-Upstream weiter.

In [11]:
async def _run_gateway_agent(task, max_steps=8, verbose=True):
    async with streamablehttp_client(GATEWAY_URL, headers=GATEWAY_HEADERS) as (read, write, _):  # <- Gateway statt stdio
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai_tools((await session.list_tools()).tools)   # <- Gateway: Schemas vom Proxy

            messages = [{"role": "user", "content": task}]
            for step in range(1, max_steps + 1):
                msg = llm(messages, tools=tools).choices[0].message

                a = {"role": "assistant", "content": msg.content}
                if msg.tool_calls:
                    a["tool_calls"] = [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls]
                messages.append(a)

                if not msg.tool_calls:
                    if verbose: print(f"[Schritt {step}] ✓ finale Antwort")
                    return msg.content

                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments or "{}")
                    if verbose: print(f"[Schritt {step}] → Gateway call_tool {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)   # <- Gateway: Ausfuehrung ueber LiteLLM
                    text = _mcp_text(result)
                    if verbose: print(f"            ⤷ {text[:90]}")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})
            return "(max_steps erreicht)"

def run_gateway_agent(task, **kw):
    return run_async(lambda: _run_gateway_agent(task, **kw))

answer = run_gateway_agent(
    "Wie spaet ist es laut Server? Rechne ausserdem 17+25, "
    "und was weiss die Wissensdatenbank ueber MCP?"
)
print("\n=== Ergebnis (über das LiteLLM-Gateway) ===\n", answer)

[Schritt 1] → Gateway call_tool demo-current_time({})
            ⤷ 2026-06-16T15:01:41
[Schritt 1] → Gateway call_tool demo-add({'a': 17, 'b': 25})
            ⤷ 42.0
[Schritt 1] → Gateway call_tool demo-search_kb({'query': 'MCP'})
            ⤷ - mcp: Das Model Context Protocol (MCP) ist ein offener Standard von Anthropic (2024), der
[Schritt 2] ✓ finale Antwort

=== Ergebnis (über das LiteLLM-Gateway) ===
 Laut Server ist es **2026-06-16T15:01:41**.

**17 + 25 = 42**

Zur Wissensdatenbank über **MCP**:
- **MCP (Model Context Protocol)** ist ein offener Standard von Anthropic (2024), der **Tools, Ressourcen und Prompts** über **JSON-RPC** zwischen Client und Server austauscht.
- Es gibt auch einen Eintrag zu einem Vortrag **„AI Agents under the Hood“**, der einen Agenten from scratch baut und mit echtem MCP-Tool-Zugriff über einen separaten Serverprozess endet.


## 5 · Interaktiv (optional) — frag den Agenten selbst, mit Gedächtnis

Wie im MCP-Notebook: eine Historie `chat_messages`, die über alle Fragen wächst — so funktionieren **Folgefragen**. Die Tools kommen weiter über das Gateway.

> Probier: *„Addiere 128 und 96"* → *„und das mal 2?"* → *„Was weiß die Wissensdatenbank über Agenten?"*
> `reset` löscht das Gedächtnis, leere Eingabe oder `exit` beendet.

In [ ]:
chat_messages = []   # <- Gedaechtnis: waechst mit jeder Frage + Antwort

async def _chat_turn(messages, max_steps=8):
    async with streamablehttp_client(GATEWAY_URL, headers=GATEWAY_HEADERS) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = to_openai_tools((await session.list_tools()).tools)
            for step in range(1, max_steps + 1):
                msg = llm(messages, tools=tools).choices[0].message
                a = {"role": "assistant", "content": msg.content}
                if msg.tool_calls:
                    a["tool_calls"] = [{"id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                        for tc in msg.tool_calls]
                messages.append(a)
                if not msg.tool_calls:
                    return msg.content
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments or "{}")
                    print(f"   → Gateway call_tool {tc.function.name}({args})")
                    result = await session.call_tool(tc.function.name, args)
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": _mcp_text(result)})
            return "(max_steps erreicht)"

print("Gateway-Agent mit Gedaechtnis bereit. ('reset' loescht, leer/'exit' beendet.)\n")
while True:
    frage = input("\U0001f64b Deine Frage (oder 'exit'): ").strip()
    if not frage or frage.lower() in ("exit", "quit", "ende"):
        print("Beendet."); break
    if frage.lower() == "reset":
        chat_messages.clear(); print("Gedaechtnis geloescht.\n"); continue
    chat_messages.append({"role": "user", "content": frage})
    antwort = run_async(lambda: _chat_turn(chat_messages))
    print("\n\U0001f916", antwort, "\n")

Gateway-Agent mit Gedaechtnis bereit. ('reset' loescht, leer/'exit' beendet.)



🙋 Deine Frage (oder 'exit'):  Gib mir alle mails aus dem CSS Ordner


   → Gateway call_tool ms365-list-mail-folders({'top': 20, 'select': 'id,displayName,parentFolderId,childFolderCount,totalItemCount,unreadItemCount'})
   → Gateway call_tool ms365-list-mail-child-folders({'mailFolderId': 'AAMkAGU4Y2QwYjMzLWMzYWMtNDdkNC1iZjQ3LTI1MTQ0YTYwMTI5ZAAuAAAAAABTXgkNFETDSbYbeGfkp8yyAQAsKS5Zxl13QI-duPPZYaiZAAe__K_mAAA=', 'top': 50, 'select': 'id,displayName,parentFolderId,childFolderCount,totalItemCount,unreadItemCount'})
   → Gateway call_tool ms365-list-mail-folder-messages({'mailFolderId': 'AAMkAGU4Y2QwYjMzLWMzYWMtNDdkNC1iZjQ3LTI1MTQ0YTYwMTI5ZAAuAAAAAABTXgkNFETDSbYbeGfkp8yyAQAsKS5Zxl13QI-duPPZYaiZAAe__K_mAAA=', 'fetchAllPages': True, 'select': 'id,subject,from,toRecipients,receivedDateTime,bodyPreview,isRead,hasAttachments', 'top': 50, 'orderby': 'receivedDateTime desc'})

🤖 Im CSS-Ordner selbst sind **keine Mails**.

Der Ordner **CSS** hat 4 Unterordner, und dort liegen die Mails:
- Christian Hartmann: 4
- Deniel Biletic: 0
- Felix Birkenbach: 11
- Winfried Si

🙋 Deine Frage (oder 'exit'):  ja aus den unterordnern


   → Gateway call_tool ms365-list-mail-folder-messages({'mailFolderId': 'AAMkAGU4Y2QwYjMzLWMzYWMtNDdkNC1iZjQ3LTI1MTQ0YTYwMTI5ZAAuAAAAAABTXgkNFETDSbYbeGfkp8yyAQAsKS5Zxl13QI-duPPZYaiZAAMhHsaKAAA=', 'fetchAllPages': True, 'select': 'id,subject,from,toRecipients,receivedDateTime,bodyPreview,isRead,hasAttachments', 'top': 50, 'orderby': 'receivedDateTime desc'})
   → Gateway call_tool ms365-list-mail-folder-messages({'mailFolderId': 'AAMkAGU4Y2QwYjMzLWMzYWMtNDdkNC1iZjQ3LTI1MTQ0YTYwMTI5ZAAuAAAAAABTXgkNFETDSbYbeGfkp8yyAQAsKS5Zxl13QI-duPPZYaiZAAe__K_oAAA=', 'fetchAllPages': True, 'select': 'id,subject,from,toRecipients,receivedDateTime,bodyPreview,isRead,hasAttachments', 'top': 50, 'orderby': 'receivedDateTime desc'})
   → Gateway call_tool ms365-list-mail-folder-messages({'mailFolderId': 'AAMkAGU4Y2QwYjMzLWMzYWMtNDdkNC1iZjQ3LTI1MTQ0YTYwMTI5ZAAuAAAAAABTXgkNFETDSbYbeGfkp8yyAQAsKS5Zxl13QI-duPPZYaiZAAe__K_pAAA=', 'fetchAllPages': True, 'select': 'id,subject,from,toRecipients,receivedDateTime,body

## 6 · Aufräumen — `docker compose down`

Alle Container (`db`, `demo-mcp`, `litellm`) stoppen und entfernen — Port `:4000` wird wieder frei. (Die DB liegt in einem Named Volume und bleibt erhalten; mit `docker compose down -v` würdest du auch die Daten löschen.)

In [ ]:
res = subprocess.run(["docker", "compose", "down"], capture_output=True, text=True)
print(res.stdout.strip() or res.stderr.strip() or "down")

## Mitnehmen — warum ein Gateway (und warum Compose)?

1. **Der Agent bleibt unberührt.** Wieder *derselbe* Loop. Es wechselt nur der **Transport** (stdio → streamable-http) und das **Ziel** (Server → LiteLLM).
2. **Ein Endpoint für n Server.** Neue Server = ein Eintrag in `mcp_servers` der `litellm_config.yaml`, kein Client-Code.
3. **Zentrale Governance.** Auth per `x-litellm-api-key`, dazu (nicht gezeigt) Zugriff pro Key/Team, Rate-Limits, Logging, Budgets — an *einer* Stelle.
4. **Namespacing inklusive.** Tool-Namen werden mit dem Server-Alias präfixt (`demo-add`) — kollisionsfrei über mehrere Server.
5. **Infrastruktur als Code.** `docker-compose.yml` ersetzt das Prozess-Management im Notebook: reproduzierbar, ein Befehl rauf/runter, und im Compose-Netz reden die Services per **Service-Name** statt über `host.docker.internal`.

> **Bonus:** LiteLLM ist primär ein **LLM-Gateway**. In der Praxis liegen damit *LLM-Zugriff* **und** *MCP-Tools* hinter demselben Tor — ein einziger Kontrollpunkt für deine Agenten-Infrastruktur.